# Bank Term Deposit Classification

## Executive Summary

A classification model to predict whether a bank client will subscribe to a term deposit, comparing Decision Tree, Random Forest, and Support Vector Classification.

### Methodology
- Data preprocessing and feature encoding
- Standardization and train-test splitting
- SMOTE-based class balancing
- Hyperparameter optimization via GridSearchCV
- Comprehensive evaluation using multiple metrics

### Evaluation Framework
Metrics: Accuracy, Precision, Recall, F1-Score


In [1]:
# Import required libraries
from pandas import read_csv, get_dummies, DataFrame
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from pandas import read_csv, get_dummies, DataFrame
# read_csv is used to load the dataset from the CSV file
# get_dummies will be used later to convert categorical data into numerical form
# DataFrame is used to create and manage tabular data during processing

from google.colab import drive
drive.mount('/content/drive')
data1 = read_csv('/content/drive/MyDrive/Z/Study/DBS/SEM 2/ML/CA1/Dataset/bank.csv')

# the dataset is loaded from Google Drive storage to avoid repeated uploads


Mounted at /content/drive


## Data Exploration

Let us examine the dataset to understand its structure and contents.


In [2]:
data1.shape
#to understand and see the data

(4521, 17)

In [3]:
data1.info()
#to find the object details and information

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        4521 non-null   int64 
 1   job        4521 non-null   object
 2   marital    4521 non-null   object
 3   education  4521 non-null   object
 4   default    4521 non-null   object
 5   balance    4521 non-null   int64 
 6   housing    4521 non-null   object
 7   loan       4521 non-null   object
 8   contact    4521 non-null   object
 9   day        4521 non-null   int64 
 10  month      4521 non-null   object
 11  duration   4521 non-null   int64 
 12  campaign   4521 non-null   int64 
 13  pdays      4521 non-null   int64 
 14  previous   4521 non-null   int64 
 15  poutcome   4521 non-null   object
 16  y          4521 non-null   object
dtypes: int64(7), object(10)
memory usage: 600.6+ KB


## Encode Categorical Variables

In this step, categorical values are converted into numerical form so that they can be used by machine learning models.

In [4]:
# encode binary categorical variables for analysis

data1['default']
data1['loan']
data1['housing']
data1['y']


,y
0,no
1,no
2,no
3,no
4,no
...,...
4516,no
4517,no
4518,no
4519,no


In [5]:
# convert binary categorical features into numeric values manually
data1['default'] = data1['default'].map({'yes': 1, 'no': 0})
data1['loan'] = data1['loan'].map({'yes': 1, 'no': 0})
data1['housing'] = data1['housing'].map({'yes': 1, 'no': 0})
data1['y'] = data1['y'].map({'yes': 1, 'no': 0})


In [6]:
data1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        4521 non-null   int64 
 1   job        4521 non-null   object
 2   marital    4521 non-null   object
 3   education  4521 non-null   object
 4   default    4521 non-null   int64 
 5   balance    4521 non-null   int64 
 6   housing    4521 non-null   int64 
 7   loan       4521 non-null   int64 
 8   contact    4521 non-null   object
 9   day        4521 non-null   int64 
 10  month      4521 non-null   object
 11  duration   4521 non-null   int64 
 12  campaign   4521 non-null   int64 
 13  pdays      4521 non-null   int64 
 14  previous   4521 non-null   int64 
 15  poutcome   4521 non-null   object
 16  y          4521 non-null   int64 
dtypes: int64(11), object(6)
memory usage: 600.6+ KB


In [7]:
data1 = get_dummies(data1,['marital', 'job', 'education', 'contact', 'month', 'poutcome'],dtype=int)
# get_dummies was imported from pandas and applied during encoding


## Separate Features and Target

Split data into X (features) and y (target).

In [8]:
# separate input features from the target variable
X = data1.drop('y', axis=1)
y = data1['y']

print(X.shape)
print(y.shape)

(4521, 48)
(4521,)


## Scale the Features ( x only )

**Why:** Different features have different ranges. Scaling makes them comparable.

In [9]:
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(X)
DataFrame(X_scaled)
# DataFrame was imported at the top while reading the data


,0,1,2,3,4,5,6,7,8,9,...,38,39,40,41,42,43,44,45,46,47
0,-1.056270,-0.130759,0.121072,-1.142051,-0.424756,0.374052,-0.711861,-0.576829,-0.407218,-0.320413,...,-0.364805,-0.104676,-0.669064,-0.306828,7.450671,-0.107869,-0.348652,-0.213447,-0.171381,0.469300
1,-0.772583,-0.130759,1.118644,0.875617,2.354292,-0.596026,-0.169194,-0.576829,2.989044,2.041734,...,-0.364805,-0.104676,1.494626,-0.306828,-0.134216,-0.107869,2.868193,-0.213447,-0.171381,-2.130831
2,-0.583458,-0.130759,-0.024144,0.875617,-0.424756,0.010273,-0.303898,-0.576829,2.899143,0.270124,...,-0.364805,-0.104676,-0.669064,-0.306828,-0.134216,-0.107869,2.868193,-0.213447,-0.171381,-2.130831
3,-1.056270,-0.130759,0.017726,0.875617,2.354292,-1.566105,-0.250017,0.387967,-0.407218,-0.320413,...,2.741190,-0.104676,-0.669064,-0.306828,-0.134216,-0.107869,-0.348652,-0.213447,-0.171381,0.469300
4,1.686036,-0.130759,-0.472753,0.875617,-0.424756,-1.323585,-0.146102,-0.576829,-0.407218,-0.320413,...,-0.364805,-0.104676,1.494626,-0.306828,-0.134216,-0.107869,-0.348652,-0.213447,-0.171381,0.469300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,-0.772583,-0.130759,-0.583410,0.875617,-0.424756,1.707910,0.250315,0.709566,-0.407218,-0.320413,...,-0.364805,-0.104676,-0.669064,-0.306828,-0.134216,-0.107869,-0.348652,-0.213447,-0.171381,0.469300
4517,1.496912,7.647669,-1.573671,0.875617,2.354292,-0.838546,-0.427057,-0.576829,-0.407218,-0.320413,...,-0.364805,-0.104676,1.494626,-0.306828,-0.134216,-0.107869,-0.348652,-0.213447,-0.171381,0.469300
4518,1.496912,-0.130759,-0.374724,-1.142051,-0.424756,0.374052,-0.434754,2.639160,-0.407218,-0.320413,...,-0.364805,-0.104676,-0.669064,-0.306828,-0.134216,-0.107869,-0.348652,-0.213447,-0.171381,0.469300
4519,-1.245394,-0.130759,-0.094925,-1.142051,-0.424756,-1.202326,-0.519426,0.387967,1.710451,1.451197,...,-0.364805,-0.104676,-0.669064,-0.306828,-0.134216,-0.107869,-0.348652,4.685001,-0.171381,-2.130831


#Data Spliting


Divide the data into training and test sets for model evaluation


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.10, random_state=100
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(4068, 48)
(453, 48)
(4068,)
(453,)


Balancing Data

Apply balancing techniques on the training set to improve model learning


In [11]:
y_train.value_counts()


,count
y,
0,3601
1,467


In [12]:
# Applying SMOTE on training data
smote = SMOTE(random_state=100)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [13]:
y_train.value_counts()


,count
y,
0,3601
1,3601


# Decision Tree Classifier
Method 1: Simple Decision Tree with manually selected hyperparameters

max_depth is used to observe baseline performance and understand model behaviour


In [14]:
DT_classifier1 = DecisionTreeClassifier(max_depth=5, random_state=100)
DT_classifier1.fit(X_train, y_train)

y_pred1 = DT_classifier1.predict(X_test)


## Evaluation The Outcome

In [15]:
Accuracy = accuracy_score(y_test, y_pred1)
print('Accuracy =', round(Accuracy*100, 2), '%')

Recall = recall_score(y_test, y_pred1)
print('Recall =', round(Recall*100, 2), '%')

Precision = precision_score(y_test, y_pred1)
print('Precision =', round(Precision*100, 2), '%')

F1 = f1_score(y_test, y_pred1)
print('F1 =', round(F1*100, 2), '%')

Conf = confusion_matrix(y_test, y_pred1)
print('Confusion_matrix =\n', Conf)


Accuracy = 80.79 %
Recall = 68.52 %
Precision = 34.58 %
F1 = 45.96 %
Confusion_matrix =
 [[329  70]
 [ 17  37]]


## Using Method 2

Optimised Decision Tree using GridSearchCV


In [16]:
DT_classifier2 = DecisionTreeClassifier(random_state=100)

depths = {'max_depth': [1,2,3,4,5,6,7,8,9,10,15,20,30]}

grid_search = GridSearchCV(
    estimator=DT_classifier2,
    param_grid=depths,
    scoring='precision',
    cv=5
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=100),
             param_grid={'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20,
                                       30]},
             scoring='precision')

Best Parameters and Model Score


In [17]:
best_depth = grid_search.best_params_
print('Best Depth =', best_depth)

best_precision = grid_search.best_score_
print('Best Precision =', round(best_precision*100, 2), '%')

Best Depth = {'max_depth': 30}
Best Precision = 90.19 %


##Best Performing Model


In [18]:
best_tree = grid_search.best_estimator_
best_tree

DecisionTreeClassifier(max_depth=30, random_state=100)

##Random Forest Classifier

Method 1: Model Training


In [19]:
RF_classifier1 = RandomForestClassifier(
    n_estimators=400,
    criterion='entropy',
    max_features='sqrt'
)  # model building

RF_classifier1.fit(X_train, y_train)  # training
y_pred1 = RF_classifier1.predict(X_test)  # testing

##Evaluation of Model Performance


In [20]:
Accuracy = accuracy_score(y_test, y_pred1)
print('Accuracy =', round(Accuracy*100, 2), '%')

Precision = precision_score(y_test, y_pred1)
print('Precision =', round(Precision*100, 2), '%')

Recall = recall_score(y_test, y_pred1)
print('Recall =', round(Recall*100, 2), '%')

F1 = f1_score(y_test, y_pred1)
print('F1 =', round(F1*100, 2), '%')


Accuracy = 88.52 %
Precision = 52.94 %
Recall = 33.33 %
F1 = 40.91 %


In [21]:
confusion_matrix(y_test, y_pred1)


array([[383,  16],
       [ 36,  18]])

##Method 2:
Modelling using Hyperparameter Tuning


In [22]:
RF_classifier2 = RandomForestClassifier(
    criterion='entropy',
    max_features='sqrt'
)  # model building

trees = {'n_estimators': [100, 200, 300, 400, 500]}

grid_search1 = GridSearchCV(
    estimator=RF_classifier2,
    param_grid=trees,
    scoring='accuracy',
    cv=5
)


In [23]:
grid_search1.fit(X_train, y_train)   # fitting : training, testing, evaluation, ranking


GridSearchCV(cv=5, estimator=RandomForestClassifier(criterion='entropy'),
             param_grid={'n_estimators': [100, 200, 300, 400, 500]},
             scoring='accuracy')

##Optimal Parameters with Model Accuracy


In [24]:
no_trees = grid_search1.best_params_
print('Best number of trees =', no_trees)

Accuracy = grid_search1.best_score_
print('Accuracy =', round(Accuracy*100, 2), '%')


Best number of trees = {'n_estimators': 400}
Accuracy = 95.49 %


##Best Model

In [25]:
best_forest = grid_search1.best_estimator_
best_forest

RandomForestClassifier(criterion='entropy', n_estimators=400)

# Support Vector Machine (SVM)

Method 1: Uing Basic SVM model


##Model Training


In [26]:
SVM_classifier1 = SVC(kernel='rbf')   # create the SVM model
SVM_classifier1.fit(X_train, y_train)   # train the model

y_pred_svm1 = SVM_classifier1.predict(X_test)   # make predictions on test data

# RBF kernel is used because the data is not linearly separable
# Tuned hyperparameters:
# C controls how much error the model allows
# Kernel is set to RBF to handle non-linear patterns


In [27]:
Accuracy = accuracy_score(y_test, y_pred_svm1)
print('Accuracy =', round(Accuracy*100, 2), '%')

Precision = precision_score(y_test, y_pred_svm1)
print('Precision =', round(Precision*100, 2), '%')

Recall = recall_score(y_test, y_pred_svm1)
print('Recall =', round(Recall*100, 2), '%')

F1 = f1_score(y_test, y_pred_svm1)
print('F1 =', round(F1*100, 2), '%')
# F1-score is used as it considers both precision and recall for better decisions


Accuracy = 85.87 %
Precision = 43.06 %
Recall = 57.41 %
F1 = 49.21 %


In [28]:
confusion_matrix(y_test, y_pred_svm1)

array([[358,  41],
       [ 23,  31]])

## Method 2: Optimised SVM using GridSearchCV

GridSearchCV was used with 5-fold cross-validation to tune the model parameters.


In [30]:
SV_classifier2 = SVC()
ker_c={'kernel':['linear','rbf'], 'C':[1,10,100]}

grid_search1 = GridSearchCV(estimator=SV_classifier2, param_grid=ker_c, scoring='accuracy', cv=5) #building: 4 components
grid_search1.fit(X_train,y_train) #fitting : training, testing, evaluation , ranking

k_c1 = grid_search1.best_params_
print(k_c1)
Accuracy = grid_search1.best_score_
print('Accuracy =',round(Accuracy*100,2),'%')

{'C': 100, 'kernel': 'rbf'}
Accuracy = 95.32 %


##Final Model Choice


In [31]:
best_svc = grid_search1.best_estimator_
best_svc

SVC(C=100)

GridSearchCV with cross validation was used to determine the best value for (C) and best kernel type to be able to control how flexible margins are and what shape the decision boundaries will take in order to keep computational costs under control.

Cross-validated hyperparameter tuning was done using GridSearchCV in order to obtain a robust and unbiased choice of parameters.

#**Final Model Summary & Comparison**

In [36]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

summary = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'SVM'],
    'Accuracy': [
        accuracy_score(y_test, DT_classifier1.predict(X_test)),
        accuracy_score(y_test, RF_classifier1.predict(X_test)),
        accuracy_score(y_test, SVM_classifier1.predict(X_test))
    ],
    'Precision': [
        precision_score(y_test, DT_classifier1.predict(X_test)),
        precision_score(y_test, RF_classifier1.predict(X_test)),
        precision_score(y_test, SVM_classifier1.predict(X_test))
    ],
    'Recall': [
        recall_score(y_test, DT_classifier1.predict(X_test)),
        recall_score(y_test, RF_classifier1.predict(X_test)),
        recall_score(y_test, SVM_classifier1.predict(X_test))
    ],
    'F1 Score': [
        f1_score(y_test, DT_classifier1.predict(X_test)),
        f1_score(y_test, RF_classifier1.predict(X_test)),
        f1_score(y_test, SVM_classifier1.predict(X_test))
    ]
})

summary

,Model,Accuracy,Precision,Recall,F1 Score
0,Decision Tree,0.807947,0.345794,0.685185,0.459627
1,Random Forest,0.885210,0.529412,0.333333,0.409091
2,SVM,0.858720,0.430556,0.574074,0.492063


Choice of Evaluation Metric:
---
Evaluation metrics for determining model performance included a number of methods to determine how well each model performed (accuracy, precision, recall, f1 score) along with a confusion matrix; accuracy was used to determine the total percentage of correct predictions for a model, precision and recall were used to analyze how effective a model was at identifying all subscribers, f1 scores were analyzed as they are an average of both precision and recall and thus may be helpful in evaluating models using unbalanced banking data. Using a variety of metrics to compare models can provide a better understanding of model performance and allow for a comparison of models based on multiple measures of success, rather than just one.

OVERFITTING AVOIDENCE MECHANISM (What would you use to avoid overfitting, and why?)
---
Several methods were used to limit overfitting:

Using train-test separation

Using cross-validation in hyperparameter tuning

Limiting the model's complexity (Decision Trees, for example by limiting the tree's depth)

Using ensemble learning with Random Forest which reduces the variance of the models

Regularization as part of SVM hyper-parameters

Only applying SMOTE on the training dataset, to avoid data leakage

All of these methods helped ensure good generalizability to new datasets and prevented the models from simply memorizing the training set.

##Final Model Recommendation Recommended Model: Random Forest Classifier

Justification:

Random forest demonstrated the best performance overall from the models tested with both the highest accuracy and the highest precision. This means that the model is making good predictions and will have few false positives which is a very important feature when dealing with banking applications where having too much customer contact can be expensive for operations. It also helps avoid the overfitting behavior seen with the decision tree by using many decision trees to make predictions, and results in predictions that are both more stable and generalizable.

The SVM achieved a better F1 score, but at the expense of a lower recall, therefore Random Forest was chosen for deployment due to its more conservative nature and higher overall accuracy. The lower recall represents a trade off that the model made between confident positive predictions, versus maximizing the total number of potential subscribers identified. Given that this trade off was acceptable for real world banking campaigns, precision and stability were valued more highly then an aggressive approach to identify all possible subscribers.

Therefore based upon this comparison, Random Forest was selected as the model for deployment based on its higher accuracy, higher precision, better resistance to over fitting, and better consistency in predicting unseen test data.

Is any model underfitting? If yes, what could be the possible reasons?
---
In the original, basic SVM configuration, there were some indications that the model was underfit (as evidenced by its performance), possibly because of the default SVM parameters and the choice of kernel. Underfitting can be caused by a model being overly restricted in what it can do, so it cannot identify all the possible decision boundaries for a given classification problem. The use of hyperparameters was necessary to allow the SVM model to perform better with GridSearchCV. The decision tree had a tendency to over fit; while the random forest demonstrated the best compromise between bias and variance.